In [ ]:
import pandas as pd
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
import torch
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt

In [ ]:
DATASETS = ""
SAVE_PATH = ""

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
train_df = pd.read_csv(f"{DATASETS}/train_preprocess.tsv", sep='\t', header=None, names=['text', 'label'])
val_df = pd.read_csv(f"{DATASETS}/valid_preprocess.tsv", sep='\t', header=None, names=['text', 'label'])
test_df = pd.read_csv(f"{DATASETS}/test_preprocess.tsv", sep='\t', header=None, names=['text', 'label'])

label2idx = {'positive': 0, 'neutral': 1, 'negative': 2}
train_df['label'] = train_df['label'].map(label2idx)
val_df['label'] = val_df['label'].map(label2idx)
test_df['label'] = test_df['label'].map(label2idx)

print(f"Training: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Testing: {len(test_df)}")

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['text_clean'] = train_df['text'].apply(clean_text)
val_df['text_clean'] = val_df['text'].apply(clean_text)
test_df['text_clean'] = test_df['text'].apply(clean_text)

In [ ]:
BERT_MODEL_NAME = 'indobenchmark/indobert-base-p1'
BERT_MAX_LEN = 128
BERT_BATCH_SIZE = 16
BERT_EPOCHS = 3
BERT_LR = 2e-5
NUM_LABELS = 3

In [ ]:
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

class SmSABertDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            max_length = self.max_len,
            padding = 'max_length',
            truncation = True,
            return_tensors  = 'pt'
        )

        return {
            'input_ids' : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'label' : torch.tensor(label, dtype=torch.long)
        }

In [ ]:
bert_train_dataset = SmSABertDataset(
    texts = train_df['text_clean'].values,
    labels = train_df['label'].values,
    tokenizer = bert_tokenizer,
    max_len = BERT_MAX_LEN
)
bert_val_dataset = SmSABertDataset(
    texts = val_df['text_clean'].values,
    labels = val_df['label'].values,
    tokenizer = bert_tokenizer,
    max_len = BERT_MAX_LEN
)
bert_test_dataset = SmSABertDataset(
    texts  = test_df['text_clean'].values,
    labels = test_df['label'].values,
    tokenizer = bert_tokenizer,
    max_len = BERT_MAX_LEN
)

bert_train_loader = DataLoader(bert_train_dataset, batch_size=BERT_BATCH_SIZE, shuffle=True)
bert_val_loader = DataLoader(bert_val_dataset, batch_size=BERT_BATCH_SIZE, shuffle=False)
bert_test_loader = DataLoader(bert_test_dataset, batch_size=BERT_BATCH_SIZE, shuffle=False)

print(f"Train batches : {len(bert_train_loader)}")
print(f"Val batches : {len(bert_val_loader)}")
print(f"Test batches : {len(bert_test_loader)}")

In [ ]:
bert_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_MODEL_NAME,
    num_labels = NUM_LABELS
)

bert_model = bert_model.to(device)

total_params = sum(p.numel() for p in bert_model.parameters())
trainable_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"Total parameter : {total_params:,}")
print(f"Parameter trainable : {trainable_params:,}")

In [ ]:
bert_optimizer = AdamW(
    bert_model.parameters(),
    lr = BERT_LR,
    weight_decay = 0.01
)

total_steps = len(bert_train_loader) * BERT_EPOCHS
warmup_steps = total_steps // 10

bert_scheduler = get_linear_schedule_with_warmup(
    bert_optimizer,
    num_warmup_steps = warmup_steps,
    num_training_steps = total_steps
)

print(f"Total steps : {total_steps}")
print(f"Warmup steps : {warmup_steps}")

In [ ]:
def training(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    all_preds = []
    all_labels = []

    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(
            input_ids = input_ids,
            attention_mask = attention_mask,
            labels = labels
        )
        loss = outputs.loss
        logits = outputs.logits
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc

In [ ]:
def evaluation(model, loader):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(
                input_ids = input_ids,
                attention_mask = attention_mask,
                labels = labels
            )

            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels

In [ ]:
bert_history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_bert_val_acc = 0

for epoch in range(1, BERT_EPOCHS + 1):
    train_loss, train_acc = training(bert_model, bert_train_loader, bert_optimizer, bert_scheduler)
    val_loss, val_acc, _, _ = evaluation(bert_model, bert_val_loader)

    bert_history['train_loss'].append(train_loss)
    bert_history['train_acc'].append(train_acc)
    bert_history['val_loss'].append(val_loss)
    bert_history['val_acc'].append(val_acc)

    if val_acc > best_bert_val_acc:
        best_bert_val_acc = val_acc
        torch.save(bert_model.state_dict(), f"{SAVE_PATH}/best_bert_model.pt")

    print(f"Epoch {epoch}/{BERT_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

print(f"Best Val Accuracy: {best_bert_val_acc:.4f}")

In [ ]:
bert_model.load_state_dict(torch.load(f"{SAVE_PATH}/best_bert_model.pt"))
_, bert_test_acc, bert_preds, bert_true = evaluation(bert_model, bert_test_loader)
print(classification_report(
    bert_true, bert_preds,
    target_names=['positive', 'neutral', 'negative']
))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bert_history['train_acc'], 'b-o', label='Train Acc', linewidth=2)
ax.plot(bert_history['val_acc'], 'r-s', label='Val Acc', linewidth=2)
ax.set_title('IndoBERT - Accuracy')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(bert_history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
ax.plot(bert_history['val_loss'], 'r-s', label='Val Loss', linewidth=2)
ax.set_title('IndoBERT - Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.show()